# Video Studio — Remote GPU Server
Run this notebook on Colab or Kaggle to start a GPU-powered background removal server.

**Steps:**
1. Run all cells
2. Copy the ngrok URL printed at the end
3. Paste it into Video Studio frontend → Remote GPU field

In [ ]:
# ── Cell 1 — Detect environment ──────────────────────────────
import os

IS_KAGGLE = os.path.exists('/kaggle')
IS_COLAB  = not IS_KAGGLE

print(f'Platform: {"Kaggle" if IS_KAGGLE else "Google Colab"}')

# Set working directory
WORK_DIR = '/kaggle/working' if IS_KAGGLE else '/content'
os.makedirs(WORK_DIR, exist_ok=True)
print(f'Work dir: {WORK_DIR}')

In [ ]:
# ── Cell 2 — Install dependencies ────────────────────────────
import subprocess

packages = [
    'fastapi',
    'uvicorn[standard]',
    'rembg[gpu]',       # GPU version of rembg
    'python-multipart', # needed for file uploads in FastAPI
    'aiofiles',
    'pyngrok',
    'onnxruntime-gpu',  # GPU inference
    'opencv-python-headless',
    'Pillow',
]

subprocess.run(
    ['pip', 'install', '-q'] + packages,
    check=True
)
print('All packages installed.')

In [ ]:
# ── Cell 3 — ngrok auth token ─────────────────────────────────
# Get your free token from https://dashboard.ngrok.com/get-started/your-authtoken
# Paste it below — this is only stored in this session, never committed

NGROK_AUTH_TOKEN = 'PASTE_YOUR_TOKEN_HERE'

from pyngrok import ngrok, conf
conf.get_default().auth_token = NGROK_AUTH_TOKEN
print('ngrok token set.')

In [ ]:
# ── Cell 4 — Write the FastAPI server ────────────────────────
server_code = '''
import uuid
import asyncio
import cv2
import numpy as np
from pathlib import Path
from fastapi import FastAPI, UploadFile, File, HTTPException, BackgroundTasks, Form
from fastapi.responses import FileResponse
from fastapi.middleware.cors import CORSMiddleware
from PIL import Image
from rembg import remove, new_session
import aiofiles

app = FastAPI(title="Video Studio Remote GPU Server")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

# Directories
UPLOAD_DIR = Path("/tmp/vs_uploads")
OUTPUT_DIR = Path("/tmp/vs_outputs")
UPLOAD_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

# Load model once at startup
print("[server] Loading U2Net model...")
SESSION = new_session("u2net")
print("[server] Model ready.")

# In-memory job store
_jobs = {}


def hex_to_rgba(hex_color: str) -> tuple:
    hex_color = hex_color.lstrip("#")
    r = int(hex_color[0:2], 16)
    g = int(hex_color[2:4], 16)
    b = int(hex_color[4:6], 16)
    return (r, g, b, 255)


def process_video(input_path, output_path, bg_color="#FFFFFF", job_id=None):
    rgba = hex_to_rgba(bg_color)
    cap = cv2.VideoCapture(str(input_path))

    fps    = cap.get(cv2.CAP_PROP_FPS) or 30
    width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(str(output_path), fourcc, fps, (width, height))

    if job_id:
        _jobs[job_id]["total"] = total

    frame_idx = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        rgb     = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        pil_img = Image.fromarray(rgb)
        result  = remove(pil_img, session=SESSION)

        background = Image.new("RGBA", result.size, rgba)
        background.paste(result, mask=result.split()[3])

        final = cv2.cvtColor(
            np.array(background.convert("RGB")),
            cv2.COLOR_RGB2BGR
        )
        writer.write(final)
        frame_idx += 1

        if job_id:
            _jobs[job_id]["progress"] = frame_idx

    cap.release()
    writer.release()


@app.get("/health")
def health():
    return {"status": "ok", "mode": "remote-gpu"}


@app.get("/api/v1/bg-removal/status/{job_id}")
def get_status(job_id: str):
    job = _jobs.get(job_id)
    if not job:
        raise HTTPException(status_code=404, detail="Job not found")
    return job


@app.post("/api/v1/bg-removal/process")
async def process(
    background_tasks: BackgroundTasks,
    file: UploadFile = File(...),
    bg_color: str = Form(default="#FFFFFF"),
):
    allowed = (".mp4", ".mov", ".avi", ".mkv")
    if not file.filename.lower().endswith(allowed):
        raise HTTPException(status_code=400, detail="Unsupported format")

    contents = await file.read()

    job_id      = str(uuid.uuid4())
    input_path  = UPLOAD_DIR / f"{job_id}_{file.filename}"
    output_path = OUTPUT_DIR / f"{job_id}_nobg.mp4"

    async with aiofiles.open(input_path, "wb") as f:
        await f.write(contents)

    _jobs[job_id] = {
        "status": "queued",
        "progress": 0,
        "total": 0,
        "filename": file.filename,
        "output": str(output_path),
        "error": None,
    }

    def run():
        try:
            _jobs[job_id]["status"] = "processing"
            process_video(input_path, output_path, bg_color, job_id)
            _jobs[job_id]["status"] = "done"
        except Exception as e:
            _jobs[job_id]["status"] = "error"
            _jobs[job_id]["error"]  = str(e)

    background_tasks.add_task(run)
    return {"job_id": job_id, "status": "queued"}


@app.get("/api/v1/bg-removal/download/{job_id}")
def download(job_id: str):
    job = _jobs.get(job_id)
    if not job:
        raise HTTPException(status_code=404, detail="Job not found")
    if job["status"] != "done":
        raise HTTPException(status_code=400, detail=f"Not ready: {job['status']}")
    return FileResponse(
        path=job["output"],
        media_type="video/mp4",
        filename=f"nobg_{job['filename']}"
    )
'''

import os
server_path = os.path.join(WORK_DIR, 'server.py')
with open(server_path, 'w') as f:
    f.write(server_code)

print(f'Server written to {server_path}')

In [ ]:
# ── Cell 5 — Start server + ngrok tunnel ─────────────────────
import threading
import uvicorn
import sys
from pyngrok import ngrok

sys.path.insert(0, WORK_DIR)

# Start FastAPI in a background thread
def run_server():
    uvicorn.run(
        'server:app',
        host='0.0.0.0',
        port=8000,
        log_level='info',
    )

thread = threading.Thread(target=run_server, daemon=True)
thread.start()

# Give server a moment to start
import time
time.sleep(3)

# Open ngrok tunnel
tunnel = ngrok.connect(8000)
public_url = tunnel.public_url

print()
print('=' * 55)
print('  ✅ GPU Server is running!')
print(f'  📋 Copy this URL into Video Studio:')
print()
print(f'     {public_url}')
print()
print('  Paste it in: Remote GPU → URL field')
print('=' * 55)

## Keep this notebook running
- Do **not** close this tab while processing
- The URL changes every time you restart
- Kaggle sessions last up to **12 hours**, Colab up to **~4 hours** on free tier
- To stop the server, just interrupt the kernel